In [8]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

# ==========================================================
# MODEL PATH
# ==========================================================


CHECKPOINT = "./V5B_Final_Merged_Model/"

# ==========================================================
# LOAD MODEL
# ==========================================================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print("=" * 100)
print("LOADING V5A MODEL")
print("=" * 100)

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)

print("Loading merged model...")
model = AutoModelForCausalLM.from_pretrained(
    CHECKPOINT,
    quantization_config=bnb_config,
    device_map="auto",
)

model.eval()

print("✅ V5A Model Loaded Successfully")



LOADING V5A MODEL
Loading tokenizer...
Loading merged model...


Loading checkpoint shards: 100%|██████████| 4/4 [00:09<00:00,  2.29s/it]

✅ V5A Model Loaded Successfully


In [9]:

# ==========================================================
# SYSTEM PROMPT for v5a
# ==========================================================
SYSTEM_PROMPT = """You are Compatifi V5A, an assistant-objective prediction model.

Your ONLY task is to predict the immediate objective that the assistant
should accomplish in the current conversation.

DO NOT generate an assistant reply.

DO NOT perform memory extraction.

DO NOT extract long-term memories.

DO NOT extract people memories.

DO NOT output personality information.

DO NOT output user goals.

DO NOT output conversation summaries.

DO NOT output any fields other than the FOUR required V5A fields.

You must output EXACTLY one JSON object containing EXACTLY these four keys:

{
  "primary_objective": "...",
  "secondary_objective": "...",
  "priority": "...",
  "reason": "..."
}

Required keys:
1. primary_objective
2. secondary_objective
3. priority
4. reason

Valid primary_objective values:
- Emotional Support
- Reduce Anxiety
- Solve Problem
- Decision Support
- Planning Assistance
- Motivation
- Information Sharing
- Maintain Rapport

Valid secondary_objective values:
- None
- Build Confidence
- Clarify Situation
- Encourage Reflection
- Suggest Next Steps
- Maintain Rapport

Valid priority values:
- High
- Medium
- Low

Rules:

1. Output EXACTLY four keys.
2. Do not output memories.
3. Do not output conversation.
4. Do not output any additional keys.
5. Do not use "Increase Confidence". Use "Build Confidence".
6. The primary objective must represent what the assistant should accomplish NOW.
7. The secondary objective supports the primary objective.
8. The reason must be based only on the provided conversation.
9. Return JSON only.
10. Do not output markdown.
11. Do not output <think>.
12. Do not output explanations outside the JSON object.

Your output must begin with { and end with }.
"""

In [ ]:
# # # ==========================================================
# # # SYSTEM PROMPT for v4a
# SYSTEM_PROMPT = """You are Compatifi V4A.

# Extract ONLY stable long-term memories about the USER.

# Keep:
# - Long-term preferences
# - Long-term goals
# - Stable personality traits
# - Persistent habits
# - Profession
# - Skills
# - Communication preferences
# - Ongoing projects
# - Persistent health conditions

# Ignore:
# - Temporary emotions
# - Greetings
# - One-time events
# - Short-term plans
# - Casual conversation

# Return ONLY valid JSON.

# Output format:

# {
#   "memories": [
#     "...",
#     "...",
#     "..."
#   ]
# }

# If nothing should be remembered:

# {
#   "memories": []
# }

# Do not generate a reply.
# Do not output people memories.
# Do not output relationship analysis.
# Do not output objectives.
# Do not output any additional fields.
# """

In [10]:

# ==========================================================
# TEST CONVERSATION for v5a
# ==========================================================



test_conversations = [

    {
        "name": "SECONDARY OBJECTIVE TEST - Interview Anxiety",
        "domain": "career",
        "relationship": "Mentor",

        "conversation": """
Mentor: How are you feeling about your interview tomorrow?

User: Honestly, I'm really nervous. I know I've prepared well,
but I keep thinking I'm going to fail.

Mentor: What makes you feel that way?

User: I keep doubting whether I'm good enough for the position.

Mentor: Your preparation has been strong, and you've already
practiced the difficult questions several times.

User: I know, but I still don't feel confident.

Mentor: Then let's focus on reminding you of what you've already
accomplished and preparing you to approach the interview calmly.

User: Yes, I think I need that.
"""
    }

]



In [ ]:
# # for v4a

# test_conversations = [

#     {
#         "name": "V4A HARD REGRESSION TEST - Mixed Long-Term and Temporary Information",

#         "domain": "career",

#         "relationship": "Career Coach",

#         "conversation": """
# Career Coach: How has your week been?

# User: Honestly, this week has been exhausting. I barely slept last night
# because I was worried about my presentation this morning.

# Career Coach: Was the presentation successful?

# User: Yes, actually. It went much better than I expected. I was nervous
# before it started, but I felt confident once I got into the technical
# details.

# Career Coach: What did you enjoy most about it?

# User: Explaining the data and presenting the technical findings.
# I've realized that I really enjoy presenting complex analysis to
# stakeholders.

# Career Coach: Is that something you want to continue doing long term?

# User: Definitely. My long-term goal is to become a Principal Data Scientist
# while remaining an individual contributor. I don't want to move into
# people management.

# Career Coach: What type of working environment helps you perform best?

# User: I need uninterrupted quiet time before noon. If I get meetings or
# Slack messages during that period, my concentration gets destroyed.
# I do my deepest technical analysis during those quiet morning hours.

# Career Coach: So mornings are important for your work?

# User: Very much. I usually block my mornings for deep technical work.

# Career Coach: How do you feel about collaborative work?

# User: I don't mind collaborating when there's a clear technical problem
# to solve. What drains me is endless alignment meetings and office
# politics. I'd much rather build the model myself and then present the
# results to stakeholders.

# Career Coach: That sounds consistent with an individual contributor path.

# User: Exactly. I want to have a large strategic impact through technical
# architecture and high-level presentations without becoming a manager.

# Career Coach: What are you currently working on?

# User: I'm building a predictive analytics system for our team. It's been
# an ongoing project for several months and I expect to keep working on
# it throughout the year.

# Career Coach: Do you have any strong preferences about communication?

# User: For routine updates, I prefer asynchronous written communication.
# I like having everything documented rather than relying on meetings.

# Career Coach: Anything else important about your career direction?

# User: Yes. I don't want to work for defense contractors or tobacco
# companies. Those industries don't align with my personal values.

# Career Coach: Understood. What about your current situation?

# User: My manager keeps scheduling a 9:30 AM standup, which conflicts
# with my focused work time. We're still discussing whether we can move
# it to an asynchronous update.

# Career Coach: Hopefully you can find a solution.

# User: I hope so. Anyway, tomorrow I'm taking the afternoon off to visit
# my family, so I won't be online after lunch.

# Career Coach: Sounds good. Enjoy the time with them.

# User: Thanks. Also, my laptop battery died yesterday, so I had to work
# from my phone for a few hours. It was incredibly annoying.

# Career Coach: That's unfortunate.

# User: Yeah, but it's fixed now. More importantly, I really want to keep
# building toward the Principal Data Scientist role.
# """
#     }

# ]

In [11]:

# ==========================================================
# RUN TESTS
# ==========================================================

for test in test_conversations:

    print("\n")
    print("=" * 100)
    print(test["name"])
    print("=" * 100)

    # for v5a
    
    user_prompt = f"""
Domain: {test['domain']}

Relationship: {test['relationship']}

Conversation:

{test['conversation']}

Instruction:
Predict the assistant objective.
"""

    
    # for v4a
#     user_prompt = f"""
# Domain: {test['domain']}

# Relationship: {test['relationship']}

# Conversation:

# {test['conversation']}

# Instruction:
# Extract long-term memories about the user.
# """
    
    
    
    
    
    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_prompt,
        },
    ]

    # ------------------------------------------------------
    # CHAT TEMPLATE
    # ------------------------------------------------------

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    # ------------------------------------------------------
    # TOKENIZE
    # ------------------------------------------------------

    inputs = tokenizer(
        text,
        return_tensors="pt",
    ).to(model.device)

    # ------------------------------------------------------
    # GENERATE
    # ------------------------------------------------------

    with torch.no_grad():

        outputs = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
        )

    # ------------------------------------------------------
    # DECODE ONLY NEW TOKENS
    # ------------------------------------------------------

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    )

    # ------------------------------------------------------
    # PRINT RESULT
    # ------------------------------------------------------

    print("\nMODEL OUTPUT")
    print("-" * 100)
    print(response)

    print("\n")




SECONDARY OBJECTIVE TEST - Interview Anxiety

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

{"primary_objective":"Reduce Anxiety","secondary_objective":"Build Confidence","priority":"High","reason":"The user is experiencing high anxiety and self-doubt before an important interview."}




====================================================================================================
SECONDARY OBJECTIVE TEST - Interview Anxiety
====================================================================================================

MODEL OUTPUT
----------------------------------------------------------------------------------------------------
<think>

</think>

{"primary_objective":"Reduce Anxiety","secondary_objective":"Increase Confidence","priority":"High","reason":"The user is feeling very nervous and doubting their readiness for an upcoming interview.","memories":null,"people":null}